# Product Pairs per Order

## Tables

**`orders1`** — records which products were ordered together under the same order.

| Column | Type |
| --- | --- |
| `order_id` | int |
| `customer_id` | int |
| `product_id` | int |

**`products1`** — maps each product ID to its name.

| Column | Type |
| --- | --- |
| `id` | int |
| `name` | varchar(10) |

## Question

For every order that contains **2 or more products**, list all **unique product pairs** (where `p1 < p2` by product ID) and report how many times each pair of product names appears across all orders.

Return the concatenated product-name pair and its `count`, ordered by `count`.

### Approach
1. **Self-join** `orders1` on `order_id` where `o1.product_id < o2.product_id` to get all unique pairs per order.
2. **Join** each pair to `products1` twice (for `p1` and `p2`) to resolve product names.
3. **Group by** the concatenated name pair and count occurrences.

In [0]:
%sql

-- ============================================================
-- Cell 1: Create and populate `orders1` and `products1` tables
-- ============================================================
create table b_sql.b_practice.orders1
(
order_id int,
customer_id int,
product_id int
);

insert into b_sql.b_practice.orders1 VALUES 
(1, 1, 1),
(1, 1, 2),
(1, 1, 3),
(2, 2, 1),
(2, 2, 2),
(2, 2, 4),
(3, 1, 5);

create table b_sql.b_practice.products1 (
id int,
name varchar(10)
);
insert into b_sql.b_practice.products1 VALUES 
(1, 'A'),
(2, 'B'),
(3, 'C'),
(4, 'D'),
(5, 'E');

In [0]:
%sql
-- ============================================================
-- Cell 2: Preview the orders1 table
-- ============================================================
select * from b_sql.b_practice.orders1;

In [0]:
%sql
-- ============================================================
-- Cell 3: Preview the products1 table
-- ============================================================
select * from b_sql.b_practice.products1

In [0]:
%sql
-- ============================================================
-- Cell 4: SQL solution — product pairs per order
-- CTE: self-join orders1 (o1.product_id < o2.product_id) for unique pairs
-- Final: join to products1 twice, group by name pair, count + order
-- ============================================================
with cte as (select o1.order_id, o1.product_id as p1, o2.product_id as p2 from b_sql.b_practice.orders1 o1 join b_sql.b_practice.orders1 o2 on o1.order_id=o2.order_id and o1.product_id<o2.product_id order by o1.order_id)
select concat_ws(',', p1.name, p2.name), count(*) as count from cte join b_sql.b_practice.products1 p1 on cte.p1=p1.id join b_sql.b_practice.products1 p2 on cte.p2=p2.id group by concat_ws(',', p1.name, p2.name) order by count

In [0]:
# ============================================================
# Cell 5: Load orders1 and products1 into PySpark DataFrames
# ============================================================
df_order=spark.read.table("b_sql.b_practice.orders1")
df_product=spark.read.table("b_sql.b_practice.products1")
df_order.display()
df_product.display()

In [0]:
# ============================================================
# Cell 6: Import PySpark SQL functions, types, and Window
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Cell 7: PySpark solution — product pairs per order
# Step 1: Self-join orders (o1.product_id < o2.product_id) for unique pairs
# Step 2: Select order_id + p1 + p2 product IDs
# Step 3: Join to products1 twice to resolve product names
# Step 4: Group by concatenated name pair, count occurrences
# ============================================================
df_join=df_order.alias("o1").join(df_order.alias("o2"),((col("o1.order_id")==col("o2.order_id")) & (col("o1.product_id")<col("o2.product_id") )))
df_oj=df_join.select(col("o1.order_id"),col("o1.product_id").alias("p1"),col("o2.product_id").alias("p2"))
df_op=df_oj.alias("o").join(df_product.alias("p1"),(col("o.p1")==col("p1.id")),"inner").join(df_product.alias("p2"),(col("o.p2")==col("p2.id")),"inner")
df_op.display()
df_final=df_op.groupBy(concat_ws("_",col("p1.name"),col("p2.name")).alias("brand")).agg(count("*").alias("cnt"))
df_final.display()